## GRADIO 

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI 
import gradio as gr 

In [ ]:
##Load the environment variales in a file called .env
# Print the key prefixes to help with any debugging

load_dotenv(override=True)
openai_api_key = os.getenv("OPEN_API_KEY")


if openai_api_key:
   print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")

In [ ]:
# Connect to OpenAI, Anthropic and Google; comment out the Claude or Google lines if you're not using them

openai = OpenAI()

In [ ]:
system_message = "You're a helpful assistant"

def message_gpt(prompt):
    messages = [{"role": "system", "content": system_message}, {"role": "user", "content": prompt}]
    response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages)
    return response.choices[0].message.content

In [ ]:
message_gpt("What is your name?")

USER INTERFACE - GRADIO

In [ ]:
def shout(text):
    print(f"shout has been called with an input{text}")
    return text.upper()

In [ ]:
shout("hello")

In [ ]:
# Adding share=True means that it can be accessed publically
# NOTE: Some Anti-virus software and Corporate Firewalls might not like you using share=True. 
# Adding inbrowser=True which can open it in dfferent browser
# Adding authenication for secure use

#u sing share=true, so you can hsare publicly
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(share=True)

# Using inbrowser=True to open in dofferent browser
gr.Interface(fn=shout,inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True)

# Adding authentication for secure use
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode= "never").launch(inbrowser=True, auth=("sk", "apples")) 

# adding dark mode 
# Define this variable and then pass js=force_dark_mode when creating the Interface

force_dark_mode = """
function refresh() {
    const url = new URL(window.location);
    if (url.searchParams.get('__theme') !== 'dark') {
        url.searchParams.set('__theme', 'dark');
        window.location.href = url.href;
    }
}
"""
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never", js=force_dark_mode).launch()

In [ ]:
# Adding some more changes

message_input = gr.Textbox(label="Your message:", info="Enter a message to be shouted", lines=7)
message_output = gr.Textbox(label="Response:", lines=8)

view = gr.Interface(
    fn=shout,
    title="Shout", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=["hello", "howdy"], 
    flagging_mode="never"
    )
view.launch()

In [ ]:
# Adding for gpt-4.1-nano 

message_input = gr.Textbox(label="Your message:", info="Enter a message for GPT-4.1-mini", lines=7)
message_output = gr.Textbox(label="Response:", lines=8)

view = gr.Interface(
    fn=message_gpt,
    title="GPT", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=["hello", "howdy"], 
    flagging_mode="never"
    )
view.launch()

In [ ]:
# using markdown in gradio


system_prompt = "You are a helpful assitant that responds in markdown without code block"

message_input= gr.Textbox(label="Your message:", info="Enter a message for GPT-5.1-nano", lines=7)
message_output = gr.Markdown(label="Response: ")

view = gr.Interface(
    fn=message_gpt,
    title="GPT", 
    inputs=[message_input],
        outputs=[message_output],
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
        ], 
        flagging_mode="never"
    )
view.launch()

In [ ]:
# Call that streams back results

def stream_gpt(prompt):
    messages=[
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ]
    stream = openai.chat.completions.create(
        model='gpt-4.1-mini',
        messages=messages,
        stream=True
    )

    result=""
    for chunk in stream:
        result+= chunk.choices[0].delta.content or ""
        yield result



In [ ]:
message_input = gr.Textbox(label="Your message:",info= "Enter a message for GPT-5.1-nano", lines=7)
message_output= gr.Markdown(label="Response: ")

view = gr.Interface(
    fn= stream_gpt,
    title="GPT",
    inputs=[message_input],
    outputs=[message_output],
    examples=[
    "Explain the Transformer architecture to a layperson",
    "Explain the Transformer architecture to an aspiring AI engineer",
        ], 
    flagging_mode="never"
    )
view.launch()


In [ ]:
!ollama pull llama3.2:1b
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

ollama_model="llama3.2:1b"

In [ ]:

def stream_ollama(prompt):
    messages =[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
    stream = ollama.chat.completions.create(
        model = ollama_model,
        messages = messages,
        stream= True
        )
    result =""
    for chunk in stream:
        result+=chunk.choices[0].delta.content or ""
        yield result

#### Sending two models: GPT and OLLAMA

In [ ]:
def stream_model(prompt, model):
    if model=="GPT":
        result = stream_gpt(prompt)
    elif model=="Ollama":
        result = stream_ollama(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result
    

In [ ]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for the LLM", lines=7)
model_selector = gr.Dropdown(["GPT", "Ollama"], label="Select model", value="GPT")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_model,
    title="LLMs", 
    inputs=[message_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["Explain the Transformer architecture to a layperson", "GPT"],
            ["Explain the Transformer architecture to an aspiring AI engineer", "Ollama"]
        ], 
    flagging_mode="never"
    )
view.launch()






## Building the company brochure using Gradio Interface

In [ ]:
from scraper import fetch_website_contents

In [ ]:
system_message =  """
You are an assistant that analyzes the contents of a company website landing page
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
"""

In [ ]:
def stream_brochure(company_name, url, model):
    yield ""
    prompt = f"Please gemerate a company brochure for {company_name}. Here is their landing page:\n"
    prompt+= fetch_website_contents(url)
    if model == "GPT":
        result = stream_gpt(prompt)
    elif model == "Ollama":
        result = stream_ollama(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result



In [ ]:
name_input = gr.Textbox(label="Company name: ")
url_input= gr.Textbox(label="Landing page URL including http:// or http://")
model_selector= gr.Dropdown(["GPT", "Ollama"], label="Select model", value="GPT")
message_output= gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_brochure,
    title="Brochure Generator", 
    inputs=[name_input, url_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["Hugging Face", "https://huggingface.co", "GPT"],
            ["Khan Academy", "https://blog.khanacademy.org/", "Ollama"]
        ], 
    flagging_mode="never"
    )
view.launch()




